In [6]:
# Установка (если ещё не ставил)
# !pip install ultralytics -U

from ultralytics import YOLO
import torch
import os
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
if torch.cuda.is_available():
    print(f'GPU доступен: {torch.cuda.get_device_name(0)}')
else:
    print('GPU не найден, используем CPU')


path = "E:/Education/4 course 2 semester/Practice/Base/Datasets/Combined_dataset/marine.yaml"

if not os.path.exists(path):
    print("Проверь путь к yaml:")
    print(os.listdir("E:/Education/4 course 2 semester/Practice/Base/Datasets/Combined_dataset"))
else:
    print("yaml найден, всё ок.")

GPU доступен: NVIDIA GeForce GTX 1660 SUPER
yaml найден, всё ок.


In [8]:
model = YOLO("yolo26n-seg.pt")

results = model.train(
    data=path,
    task="segment",
    epochs=75,
    imgsz=512,
    batch=8,            
    lr0=1e-4,
    optimizer="AdamW",
    device=0,           
    project="semantic_segmentation",
    name="training",
    exist_ok=True,
)

New https://pypi.org/project/ultralytics/8.4.31 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.11.15 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce GTX 1660 SUPER, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:/Education/4 course 2 semester/Practice/Base/Datasets/Combined_dataset/marine.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=75, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26

In [9]:
metrics = model.val()

print("\nОсновные метрики:")
print("mAP (50-95):",  metrics.box.map)      # если box есть
print("mAP@0.5:",      metrics.box.map50)

# если есть seg, то:
if hasattr(metrics, "seg"):
    print("mask mAP (50-95):",  metrics.seg.map)
    print("mask mAP@0.5:",       metrics.seg.map50)

Ultralytics 8.4.26  Python-3.11.15 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce GTX 1660 SUPER, 6144MiB)
YOLO26n-seg summary (fused): 139 layers, 2,689,469 parameters, 0 gradients, 9.0 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2371.9574.4 MB/s, size: 311.8 KB)
val: Scanning E:\Education\4 course 2 semester\Practice\Base\Datasets\Combined_dataset\labels\val.cache... 198 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 198/198  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 3.7it/s 3.5s0.1s
                   all        198       1791      0.677      0.431      0.444      0.336      0.649      0.408      0.419      0.313
                 water        198        539      0.771      0.362      0.406      0.358      0.748      0.351      0.393      0.346
              obstacle        198        761      0.582      0.486       0.43      0.258      0.529      0.439  

In [10]:
val_dir = "E:/Education/4 course 2 semester/Practice/Base/Datasets/Combined_dataset/images/val"

img_paths = sorted(glob.glob(os.path.join(val_dir, "*.jpg")))[:5]

print(f"Найдено {len(img_paths)} изображений.")

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for i, img_path in enumerate(img_paths):
    img = Image.open(img_path)
    results = model(img_path, imgsz=512, augment=False)
    axes[i].imshow(results[0].plot(boxes=False, img=img, pil=True))
    axes[i].set_title(f"Image {i+1}", fontsize=10)
    axes[i].axis("off")

axes[5].remove()

plt.tight_layout()
plt.show()

Найдено 5 изображений.

image 1/1 E:\Education\4 course 2 semester\Practice\Base\Datasets\Combined_dataset\images\val\lars_val_davimar_seq_08_00008.jpg: 288x512 1 obstacle, 2 skys, 17.9ms
Speed: 1.2ms preprocess, 17.9ms inference, 1.2ms postprocess per image at shape (1, 3, 288, 512)

image 1/1 E:\Education\4 course 2 semester\Practice\Base\Datasets\Combined_dataset\images\val\lars_val_davimar_seq_08_00037.jpg: 288x512 1 obstacle, 1 sky, 16.1ms
Speed: 1.1ms preprocess, 16.1ms inference, 1.1ms postprocess per image at shape (1, 3, 288, 512)

image 1/1 E:\Education\4 course 2 semester\Practice\Base\Datasets\Combined_dataset\images\val\lars_val_davimar_seq_36_00024.jpg: 288x512 1 obstacle, 2 skys, 17.1ms
Speed: 1.1ms preprocess, 17.1ms inference, 1.1ms postprocess per image at shape (1, 3, 288, 512)

image 1/1 E:\Education\4 course 2 semester\Practice\Base\Datasets\Combined_dataset\images\val\lars_val_davimar_seq_36_00062.jpg: 288x512 3 obstacles, 19.7ms
Speed: 1.5ms preprocess, 19.7ms in

<Figure size 1500x800 with 5 Axes>